# 🎓 Train LLM-JEPA for Symbolic Regression

This notebook trains the LLM-JEPA model on synthetic physics equations.

**What this does:**
- Clones/pulls the repository
- Syncs to Google Drive (SymbolicRegression folder)
- Downloads AI Feynman dataset if needed
- Trains with TensorBoard monitoring
- Saves checkpoints to Drive

**Runtime:** T4 GPU or better recommended

---

In [ ]:
# @title 📦 Setup: Clone Repo & Sync to Google Drive

import os
import subprocess
import yaml
from pathlib import Path
from google.colab import drive

# Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Define paths
DRIVE_FOLDER = "/content/drive/MyDrive/SymbolicRegression"
WORK_DIR = "/content/GSOC-LM-JEPA_for_Symbolic_Regression"
REPO_URL = "https://github.com/udohchuks/GSOC-LM-JEPA_for_Symbolic_Regression.git"

# Create Drive folder if not exists
Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
print(f"✅ Drive folder ready: {DRIVE_FOLDER}")

# Clone or pull repository
if os.path.exists(WORK_DIR):
    print("📦 Repository found, using existing copy")
    # Stash any local changes before pulling
    print("📦 Cloning repository...")
    os.chdir("/content")
    os.chdir(WORK_DIR)

# Sync to Drive (copy working directory)
print("🔄 Syncing to Google Drive...")
sync_target = f"{DRIVE_FOLDER}/code"
Path(sync_target).mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-av", "--delete", f"{WORK_DIR}/", f"{sync_target}/"], check=True)
print(f"✅ Synced to: {sync_target}")

# Install dependencies
print("📦 Installing dependencies...")
os.chdir(WORK_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✅ Dependencies installed")

# Create directories in Drive
cache_dir = f"{DRIVE_FOLDER}/cache"
checkpoints_dir = f"{DRIVE_FOLDER}/checkpoints"
logs_dir = f"{DRIVE_FOLDER}/tb_logs"
data_dir = f"{DRIVE_FOLDER}/Feynman_with_units"

for d in [cache_dir, checkpoints_dir, logs_dir]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"✅ Directories ready:")
print(f"   - Cache: {cache_dir}")
print(f"   - Checkpoints: {checkpoints_dir}")
print(f"   - Logs: {logs_dir}")

# Check for synthetic data
print("\n" + "=" * 60)
print("⚠️  DEPENDENCY CHECK")
print("=" * 60)

# Read cache path from config
with open(f"{WORK_DIR}/configs/small.yaml", 'r') as f:
    config = yaml.safe_load(f)
synthetic_cache = Path(config['data']['synthetic_cache'])

if synthetic_cache.exists() and len(list(synthetic_cache.glob("*.pt"))) > 0:
    n_files = len(list(synthetic_cache.glob("*.pt")))
    print(f"✅ Synthetic data found: {n_files} files")
    print(f"   Location: {synthetic_cache}")
    print("   You can proceed with training.")
else:
    print("❌ Synthetic data NOT found!")
    print(f"   Expected location: {synthetic_cache}")
    print()
    print("📋 You must run '01_generate_synthetic_data.ipynb' first!")
    print()
    print("   Steps:")
    print("   1. Open: 01_generate_synthetic_data.ipynb")
    print("   2. Generate at least some synthetic data")
    print("   3. Return here and continue with training")
    print()
    print("⚠️  Training will fail without synthetic data!")

In [ ]:
# @title 📥 Download AI Feynman Dataset (if needed)

import tarfile
import urllib.request
from pathlib import Path

# Setup AI Feynman data in Drive
print("📦 Setting up AI Feynman dataset in Drive...")
local_data = Path(f"{WORK_DIR}/data/Feynman_with_units")
drive_data = Path(f"{DRIVE_FOLDER}/Feynman_with_units")

if not drive_data.exists():
    if local_data.exists():
        print(f"   📥 Copying from local: {local_data}")
        import shutil
        shutil.copytree(local_data, drive_data)
        print(f"   ✅ Copied to Drive: {drive_data}")
    else:
        print("   ⚠️  Local AI Feynman data not found!")
        print("   Downloading from Dropbox...")
        
        # Download
        tar_path = Path(f"{DRIVE_FOLDER}/Feynman_with_units.tar.gz")
        if not tar_path.exists():
            url = "https://www.dropbox.com/s/7kgfr00qpokgz8w/Feynman_with_units.tar.gz?dl=1"
            print(f"   Downloading: {url}")
            urllib.request.urlretrieve(url, tar_path)
            print(f"   ✅ Downloaded: {tar_path}")
        
        # Extract
        print(f"   Extracting to: {drive_data}")
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(path=DRIVE_FOLDER)
        print(f"   ✅ Extracted to: {drive_data}")
else:
    print(f"   ✅ AI Feynman data already in Drive: {drive_data}")

# Update config to use Drive paths
import yaml
with open(f"{WORK_DIR}/configs/small.yaml", 'r') as f:
    config = yaml.safe_load(f)

config['data']['data_dir'] = str(drive_data) + '/'
config['data']['csv_path'] = str(Path(DRIVE_FOLDER) / 'FeynmanEquations.csv')

# Copy CSV if needed
local_csv = Path(f"{WORK_DIR}/data/FeynmanEquations.csv")
drive_csv = Path(config['data']['csv_path'])
if not drive_csv.exists() and local_csv.exists():
    import shutil
    shutil.copy(local_csv, drive_csv)
    print(f"   ✅ Copied CSV to Drive: {drive_csv}")

with open(f"{WORK_DIR}/configs/small.yaml", 'w') as f:
    yaml.dump(config, f)

print(f"✅ Config updated:")
print(f"   data_dir: {config['data']['data_dir']}")
print(f"   csv_path: {config['data']['csv_path']}")

data_dir = Path(f"{DRIVE_FOLDER}/Feynman_with_units")

if data_dir.exists() and len(list(data_dir.glob("*"))) > 10:
    print(f"✅ AI Feynman dataset already exists: {data_dir}")
    print(f"   Found {len(list(data_dir.glob('*')))} files")
else:
    print("📥 Downloading AI Feynman dataset...")
    data_dir.mkdir(parents=True, exist_ok=True)
    
    # Download from Dropbox
    tar_url = "https://www.dropbox.com/s/7kgfr00qpokgz8w/Feynman_with_units.tar.gz?dl=1"
    tar_path = f"{DRIVE_FOLDER}/Feynman_with_units.tar.gz"
    
    try:
        urllib.request.urlretrieve(tar_url, tar_path)
        print("📦 Extracting...")
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(path=str(data_dir.parent))
        print(f"✅ Dataset extracted to: {data_dir}")
        
        # Cleanup
        os.remove(tar_path)
    except Exception as e:
        print(f"⚠️ Download failed: {e}")
        print("   Please download manually from the repository")

# Preprocess AIF dataset for training
print("\n🔄 Preprocessing AI Feynman dataset...")
%cd $WORK_DIR
!python -m data.preprocess_aif --config {CONFIG_FILE}
print("✅ AIF preprocessing complete!")


In [ ]:
# @title 🔧 Force Update Config in Drive (RUN THIS FIRST!)

# This ensures the config in Drive matches the local config
import shutil, yaml
from pathlib import Path

drive_config = Path(f"{DRIVE_FOLDER}/code/configs/small.yaml")
local_config = Path(f"{WORK_DIR}/configs/small.yaml")

print(f"📄 Local config: {local_config}")
print(f"📄 Drive config: {drive_config}")
print()

# Force copy local to Drive
if local_config.exists():
    drive_config.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(local_config, drive_config)
    print(f"✅ Config copied to Drive")
    
    # Verify and display config
    with open(drive_config) as f:
        cfg = yaml.safe_load(f)
    
    d_model = cfg["model"]["d_model"]
    n_heads = cfg["model"]["n_heads"]
    n_enc = cfg["model"]["n_enc_layers"]
    n_dec = cfg["model"]["n_dec_layers"]
    bottleneck_dim = int(d_model * cfg["model"]["predictor"]["pred_bottleneck_ratio"])
    pred_n_heads = cfg["model"]["predictor"]["pred_n_heads"]
    
    print()
    print(f"✅ Config validated:")
    print(f"   d_model={d_model}, n_heads={n_heads} → head_dim={d_model//n_heads}")
    print(f"   bottleneck_dim={bottleneck_dim}, pred_n_heads={pred_n_heads} → {bottleneck_dim//pred_n_heads}")
    print(f"   n_enc_layers={n_enc}, n_dec_layers={n_dec}")
    print()
    
    # Check divisibility
    assert d_model % n_heads == 0, f"d_model {d_model} not divisible by n_heads {n_heads}"
    assert bottleneck_dim % pred_n_heads == 0, f"bottleneck_dim {bottleneck_dim} not divisible by pred_n_heads {pred_n_heads}"
    print(f"✅ All dimension checks passed!")
    print()
    print(f"📊 Expected parameters: ~991K (d_model={d_model})")
    print(f"   If you see ~515K, d_model=64 is being used (WRONG!)")
    print(f"   If you see ~991K, d_model=80 is being used (CORRECT!)")
else:
    print(f"❌ Local config not found: {local_config}")


In [ ]:
# @title ⚙️ Training Configuration

# @markdown ### Select Configuration
CONFIG_FILE = "configs/small.yaml"  # @param {type: "string"}
# @markdown - `configs/small.yaml` - ~1M params, 25k-50k equations (RECOMMENDED)
# @markdown - `configs/base_config.yaml` - ~3.4M params, 100k+ equations

# @markdown ### Training Parameters
MAX_EPOCHS = 15  # @param {type: "integer"}
BATCH_SIZE = 64  # @param {type: "integer"}
LEARNING_RATE = 5e-4  # @param {type: "number"}
USE_SYNTHETIC = True  # @param {type: "boolean"}

# @markdown ### Checkpoint & Logging
EXPERIMENT_NAME = "llmjepa_small"  # @param {type: "string"}
RESUME_FROM_CHECKPOINT = True  # Always resume from latest checkpoint in Drive

print(f"✅ Configuration: {CONFIG_FILE}")
print(f"   Epochs: {MAX_EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}")
print(f"   Experiment: {EXPERIMENT_NAME}")
print(f"   Checkpoints: {DRIVE_FOLDER}/checkpoints")
print(f"   Resume: {RESUME_FROM_CHECKPOINT} (auto-resume from latest)")
print()
print("📋 Note: Training command will use these parameters")

# Validate config dimensions
import yaml
with open(CONFIG_FILE, "r") as f:
    cfg = yaml.safe_load(f)

# Ensure d_model is divisible by n_heads
d_model = cfg["model"]["d_model"]
n_heads = cfg["model"]["n_heads"]
if d_model % n_heads != 0:
    print(f"⚠️  Fixing config: d_model {d_model} not divisible by n_heads {n_heads}")
    cfg["model"]["d_model"] = 64  # Use valid value
    n_heads = 4
    with open(CONFIG_FILE, "w") as f:
        yaml.dump(cfg, f)
    print(f"✅ Fixed: d_model=64, n_heads=4")

# Ensure bottleneck_dim is divisible by pred_n_heads
bottleneck_ratio = cfg["model"]["predictor"]["pred_bottleneck_ratio"]
pred_n_heads = cfg["model"]["predictor"]["pred_n_heads"]
bottleneck_dim = int(cfg["model"]["d_model"] * bottleneck_ratio)
if bottleneck_dim % pred_n_heads != 0:
    print(f"⚠️  Fixing config: bottleneck_dim {bottleneck_dim} not divisible by pred_n_heads {pred_n_heads}")
    cfg["model"]["predictor"]["pred_bottleneck_ratio"] = 0.25  # 64 * 0.25 = 16
    with open(CONFIG_FILE, "w") as f:
        yaml.dump(cfg, f)
    print(f"✅ Fixed: bottleneck_ratio=0.25 (bottleneck_dim=16)")

# Sync updated config to Drive
import shutil, yaml
drive_config = Path(f"{DRIVE_FOLDER}/code/configs/small.yaml")
local_config = Path(f"{WORK_DIR}/configs/small.yaml")
if local_config.exists():
    drive_config.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(local_config, drive_config)
    print(f"✅ Config synced to Drive: {drive_config}")
    
    # Verify dimensions
    with open(drive_config) as f:
        cfg = yaml.safe_load(f)
    d_model = cfg["model"]["d_model"]
    n_heads = cfg["model"]["n_heads"]
    bottleneck_dim = int(d_model * cfg["model"]["predictor"]["pred_bottleneck_ratio"])
    pred_n_heads = cfg["model"]["predictor"]["pred_n_heads"]
    
    assert d_model % n_heads == 0, f"d_model {d_model} not divisible by n_heads {n_heads}"
    assert bottleneck_dim % pred_n_heads == 0, f"bottleneck_dim {bottleneck_dim} not divisible by pred_n_heads {pred_n_heads}"
    print(f"✅ Config validated: d_model={d_model}, n_heads={n_heads}, bottleneck_dim={bottleneck_dim}, pred_n_heads={pred_n_heads}")


In [ ]:
# @title 🚀 Start Training

import time

print(f"🎯 Starting training...")
print(f"   Experiment: {EXPERIMENT_NAME}")
print(f"   Epochs: {MAX_EPOCHS}")
print(f"   Using synthetic data: {USE_SYNTHETIC}")
print(f"   Config: {CONFIG_FILE}")
        
# Sync config to Drive
import shutil
drive_config = Path(f"{DRIVE_FOLDER}/code/configs/small.yaml")
local_config = Path(f"{WORK_DIR}/configs/small.yaml")
if local_config.exists():
    drive_config.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(local_config, drive_config)
    print(f"✅ Config synced to Drive")
print("=" * 60)

# Ensure Drive cache directory exists
drive_cache_dir = Path(f"{DRIVE_FOLDER}/cache")
drive_cache_dir.mkdir(parents=True, exist_ok=True)

# Check what's already in Drive cache
aif_cache = drive_cache_dir / "aif_preprocessed.pt"
synthetic_parts = list(drive_cache_dir.glob("synthetic_*/part_*.pt"))

if aif_cache.exists():
    size_mb = aif_cache.stat().st_size / 1024 / 1024
    print(f"   ✅ AIF cache found: {aif_cache.name} ({size_mb:.1f} MB)")
else:
    print(f"   ℹ️  AIF cache will be generated")

if synthetic_parts:
    print(f"   ✅ Synthetic data found: {len(synthetic_parts)} parts")
else:
    print(f"   ℹ️  No synthetic data in Drive yet")
print()
print(f"✅ Drive cache directory: {drive_cache_dir}")

# Use Drive cache for AIF data
import yaml
with open(CONFIG_FILE, 'r') as f:
    config = yaml.safe_load(f)

# Update config to use Drive cache
config['data']['cache_dir'] = str(drive_cache_dir) + '/'
with open(CONFIG_FILE, 'w') as f:
    yaml.dump(config, f)
print(f"✅ Config updated: cache_dir = {config['data']['cache_dir']}")
print()

start_time = time.time()

# Launch training
%cd $WORK_DIR
!python -m training.train --config {CONFIG_FILE}

elapsed = time.time() - start_time
hours = elapsed / 3600

# Sync cache back to Drive (already using Drive cache, but ensure sync)
print(f"\n✅ Cache is in Drive: {drive_cache_dir}")

print("\n" + "=" * 60)
print(f"✅ Training complete!")
print(f"   Time elapsed: {hours:.2f} hours ({elapsed:.0f} seconds)")
print(f"   Checkpoints saved to: {DRIVE_FOLDER}/checkpoints")
print(f"   TensorBoard logs: {DRIVE_FOLDER}/tb_logs")
print()
print("📊 Next: Run the '📊 Open TensorBoard' cell to view training progress")

---
### 📊 View Training Progress with TensorBoard

Run this cell **during or after** training to monitor progress in real-time.

In [ ]:
# @title 📊 Open TensorBoard

# Load TensorBoard extension
%load_ext tensorboard

# Open TensorBoard
log_dir = f"{DRIVE_FOLDER}/tb_logs"
print(f"📊 Opening TensorBoard...")
print(f"   Logs directory: {log_dir}")
print()
print("TensorBoard will open below. Metrics available:")
print("   - Training loss")
print("   - Validation loss")
print("   - Learning rate")
print("   - Gradient norm")
print("   - Throughput (samples/sec)")
print()
%tensorboard --logdir {log_dir}

In [ ]:
# @title 💾 List Checkpoints

from pathlib import Path

ckpt_dir = Path(f"{DRIVE_FOLDER}/checkpoints")

if ckpt_dir.exists():
    ckpts = sorted(ckpt_dir.glob("*.ckpt"), key=lambda x: x.stat().st_mtime, reverse=True)
    
    if ckpts:
        print(f"📁 Found {len(ckpts)} checkpoints:")
        for i, ckpt in enumerate(ckpts[:10]):  # Show latest 10
            size_mb = ckpt.stat().st_size / (1024**2)
            print(f"   {i+1}. {ckpt.name} ({size_mb:.1f} MB)")
        
        if len(ckpts) > 10:
            print(f"   ... and {len(ckpts) - 10} more")
    else:
        print("⚠️ No checkpoints found yet")
else:
    print("⚠️ Checkpoint directory not found")

---
## Next Steps

After training completes:

1. **Evaluate**: Use [`03_evaluate_model.ipynb`](03_evaluate_model.ipynb) to test on AI Feynman benchmark
2. **Inference**: Use `predict.py` to generate formulas for custom data

## Tips

- **Monitor with TensorBoard**: Keep the TensorBoard tab open to watch training in real-time
- **Early stopping**: Stop training early if validation loss plateaus
- **Checkpoints saved**: All checkpoints persist in `SymbolicRegression/checkpoints/`
- **Resume training**: Set `RESUME_FROM_CHECKPOINT = True` and provide checkpoint path